# Black History Factory

Run the cells in order, top to bottom. If Colab disconnects, just reconnect and re-run from Cell 1 -- the factory resumes automatically from its last checkpoint.

## Cell 1 -- Mount Drive and create the folder tree

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
base = "/content/drive/MyDrive/BLACK_HISTORY_FACTORY"
for p in ["00_CONFIG","01_TOPICS","02_RESEARCH/raw","02_RESEARCH/verified",
          "02_RESEARCH/sources","03_SCRIPTS/narration","03_SCRIPTS/scenes",
          "04_IMAGES/generating","04_IMAGES/completed","05_AUDIO/narration",
          "05_AUDIO/music","06_VIDEOS/rendering","06_VIDEOS/completed",
          "07_THUMBNAILS","08_STATUS","09_LOGS"]:
    os.makedirs(f"{base}/{p}", exist_ok=True)
print("Drive mounted and folder tree ready at", base)

## Cell 2 -- Get the latest factory code from GitHub

In [ ]:
REPO_URL = "https://github.com/jonbBla/black-history-factory.git"  # ← edit this

import os
if not os.path.exists("/content/black-history-factory"):
    !git clone {REPO_URL} /content/black-history-factory
else:
    !cd /content/black-history-factory && git pull

import sys
sys.path.insert(0, "/content/black-history-factory")
!pip install -q -r /content/black-history-factory/requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null

## Cell 3 -- Load configuration

In [ ]:
from factory.config import Config
config = Config.load(base)
print(config.values)

## Cell 4 -- Load models

In [ ]:
USE_QWEN = True   # → research, fact-check, narration, visual bible, scenes
USE_FLUX = True   # → real scene images (either backend below; needs a GPU)
USE_PIPER = True  # → real narration audio (needs a downloaded .onnx voice)
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx -O {PIPER_VOICE_PATH}
!wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json -O {PIPER_VOICE_PATH}.json

# Model size recommendation for a free-tier T4 (14.56GB VRAM): the
# combination below (Qwen 3B + SDXL-Lightning) totals ~13GB even at full
# precision, no quantization required -- real margin instead of a razor's
# edge.
QWEN_MODEL = "Qwen/Qwen2.5-3B-Instruct"  # ← try "Qwen/Qwen2.5-7B-Instruct" if you have more VRAM

PIPER_VOICE_PATH = "/content/en_US-lessac-medium.onnx"

if USE_PIPER:
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx -O {PIPER_VOICE_PATH}
    !wget -q https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx.json -O {PIPER_VOICE_PATH}.json

models = {}

if USE_QWEN:
    from factory.qwen_client import QwenClient
    models["qwen"] = QwenClient.load(QWEN_MODEL)
    print("Qwen loaded.")

if USE_FLUX:
    # SDXL-Lightning is the recommended default: ~7GB vs FLUX's ~34GB,
    # fits a T4 without quantization tricks. Switch to FLUX (stronger
    # prompt adherence, much heavier) with:
    #   from factory.image_engine import load_flux
    #   models["flux"] = load_flux()
    from factory.image_engine import load_sdxl_lightning
    models["flux"] = load_sdxl_lightning()
    print("SDXL-Lightning loaded.")

if USE_PIPER:
    from factory.audio_engine import load_piper_voice
    models["piper"] = load_piper_voice(PIPER_VOICE_PATH)
    print("Piper voice loaded.")

if not models:
    print("Running without any loaded model -- every stage will write clearly-labeled placeholders.")

## Cell 5 -- Check for a previous in-progress job

In [ ]:
from factory.drive import DrivePaths
from factory.checkpoint import find_in_progress_job

paths = DrivePaths(root=base)
in_progress = find_in_progress_job(paths)
if in_progress:
    print(f"Resuming in-progress job: {in_progress}")
else:
    print("No in-progress job found -- will start a new one.")

## Cell 6 -- Start the factory
`GH_TOKEN` is read from Colab's Secrets panel so it's never hard-coded here. Leave it blank to skip pushing status to GitHub.

In [ ]:
from google.colab import userdata
try:
    gh_token = userdata.get('GH_TOKEN')
except Exception:
    gh_token = ""

from factory import main
main.run(max_jobs=1, gh_token=gh_token, models=models)

## Cell 7 -- Display current progress

In [ ]:
import json
from factory.utils import read_json
print(json.dumps(read_json(paths.status_current, default={}), indent=2))